1. Importing Necessary Libraries

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import Input, Model, Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, GlobalAveragePooling2D, Dropout, concatenate, Bidirectional, LSTM
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix

2. Configuration

In [2]:
DATA_PATH = r''  # Set project root
CSV_FILENAME = '../../annotations/train_detector_annotations.csv'  # merged annotations file (clean + adversarial)
IMG_SIZE = 50
NUM_CHANNELS = 3   # 3 for color images; set to 1 only if you explicitly load grayscale
NUM_CLASSES = 2    # binary detection: clean (0) / adversarial (1)
BATCH_SIZE = 64
SEED = 42
NUM_EPOCHS = 30
np.random.seed(SEED)
tf.random.set_seed(SEED)

3. Load Annotations CSV

In [3]:
# Resolve the annotations CSV path (prefer DATA_PATH if provided).
# Fail fast with a clear error if the file is missing.
# Read the CSV with auto-detected separator (handles ',', ';', '\t').
# Print a brief summary (path, rows, columns) and preview the head.

csv_path = CSV_FILENAME
if DATA_PATH:
    candidate = os.path.join(DATA_PATH, CSV_FILENAME)
    if os.path.exists(candidate):
        csv_path = candidate

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Annotations CSV not found at: {csv_path}. Place the file or update CSV_FILENAME/DATA_PATH.")

df = pd.read_csv(csv_path, sep=None, engine='python')
print('Loaded CSV:', csv_path)
print('Rows:', len(df))
print('Columns:', df.columns.tolist())
df.head()

Loaded CSV: ../../annotations/train_detector_annotations.csv
Rows: 155478
Columns: ['image_id', 'label', 'isNight', 'split', 'adv_label']


,image_id,label,isNight,split,adv_label
0,..\..\datasets\processed_images\train\img_0.jpg,1,0,train,0
1,..\..\datasets\processed_images\train\img_1.jpg,1,0,train,0
2,..\..\datasets\processed_images\train\img_2.jpg,1,0,train,0
3,..\..\datasets\processed_images\train\img_3.jpg,1,0,train,0
4,..\..\datasets\processed_images\train\img_4.jpg,1,0,train,0


3. Build X/y for binary detection

In [4]:
# Ensure we have a binary target `adv_label`:
# clean = 0, adversarial = 1
# If it's missing, infer it from `source` or `image_id`.
# Decide which column holds image paths (`image_path` or `image_id`).
# Robustly resolve image paths relative to DATA_PATH when needed.
# Load, resize, and normalize images to [0,1]; coerce to 3 channels.
# Collect optional `isNight` as a 1D aux feature.
# Return (images, aux, labels) per split: train/val/test.

if 'adv_label' not in df.columns:
    if 'source' in df.columns:
        df['adv_label'] = df['source'].apply(lambda s: 0 if str(s).strip().lower() == 'train_detector_annotations' else 1)
        print('Derived adv_label from source column.')
    elif 'image_id' in df.columns:
        df['adv_label'] = df['image_id'].apply(lambda p: 1 if 'adv_outputs' in str(p) else 0)
        print('Heuristically derived adv_label from image_id paths.')
    else:
        raise RuntimeError('No adv_label, source, or image_id column found to infer adversarial labels.')

if 'image_path' in df.columns:
    image_col = 'image_path'
elif 'image_id' in df.columns:
    image_col = 'image_id'
else:
    image_col = df.columns[0]
print('Using image column:', image_col)

def resolve_image_path(p):
    """Normalize/resolve an image path; try absolute, relative, and under DATA_PATH."""
    if not isinstance(p, str):
        return p
    p_norm = os.path.normpath(p)
    if os.path.isabs(p_norm) and os.path.exists(p_norm):
        return p_norm
    # FIX adversarial paths
    if p_norm.startswith("adv_outputs") or p_norm.startswith(r"adv_outputs"):
        p_norm = os.path.normpath(os.path.join("..", "..", "datasets", p_norm))
        if os.path.exists(p_norm):
            return p_norm
    if DATA_PATH:
        cand = os.path.join(DATA_PATH, p_norm)
        if os.path.exists(cand):
            return cand
        cand2 = os.path.join(DATA_PATH, os.path.basename(p_norm))
        if os.path.exists(cand2):
            return cand2
    return p_norm

def build_Xy_binary(df_part, img_size=IMG_SIZE, num_channels=NUM_CHANNELS):
    """Load images + isNight, normalize/resize, and return (X_img, X_aux, y) arrays."""
    X_images, X_isNight, y = [], [], []
    missing_count = 0
    unreadable_count = 0
    for _, row in tqdm(df_part.iterrows(), total=len(df_part)):
        raw = str(row[image_col])
        img_path = raw.replace('\\','/')
        img_path = resolve_image_path(img_path)
        adv_lbl = int(row['adv_label'])
        is_night = int(row.get('isNight', 0))
        if not os.path.exists(img_path):
            missing_count += 1
            if missing_count <= 5:
                print('Missing image:', img_path)
            continue
        img = cv2.imread(img_path)
        if img is None:
            unreadable_count += 1
            if unreadable_count <= 5:
                print('Could not read image:', img_path)
            continue
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        if img.shape[2] == 4 and num_channels == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        img = cv2.resize(img, (img_size, img_size)).astype('float32') / 255.0
        X_images.append(img)
        X_isNight.append(is_night)
        y.append(adv_lbl)
    if len(X_images) == 0:
        raise RuntimeError('No images loaded — verify paths and CSV.')
    X_images = np.array(X_images)
    X_isNight = np.array(X_isNight).reshape(-1,1)
    y = np.array(y).astype('int32')
    return X_images, X_isNight, y

X_train_img, X_train_meta, y_train = build_Xy_binary(df[df['split']=='train'])
X_val_img,   X_val_meta,   y_val   = build_Xy_binary(df[df['split']=='val'])
X_test_img,  X_test_meta,  y_test  = build_Xy_binary(df[df['split']=='test'])

print('Shapes:')
print('X_train_img', X_train_img.shape, 'X_train_meta', X_train_meta.shape, 'y_train', y_train.shape)
print('X_val_img', X_val_img.shape, 'X_val_meta', X_val_meta.shape, 'y_val', y_val.shape)
print('X_test_img', X_test_img.shape, 'X_test_meta', X_test_meta.shape, 'y_test', y_test.shape)


Using image column: image_id


 32%|███▏      | 6438/20058 [00:01<00:03, 4295.22it/s]

Missing image: ..\..\datasets\adv_outputs\ANN_fgsm\processed_images\val\img_30707.png
Missing image: ..\..\datasets\adv_outputs\ANN_fgsm\processed_images\val\img_30708.png
Missing image: ..\..\datasets\adv_outputs\ANN_fgsm\processed_images\val\img_30709.png
Missing image: ..\..\datasets\adv_outputs\ANN_fgsm\processed_images\val\img_30710.png
Missing image: ..\..\datasets\adv_outputs\ANN_fgsm\processed_images\val\img_30711.png


 33%|███▎      | 14358/43299 [00:03<00:06, 4262.11it/s]

Missing image: ..\..\datasets\adv_outputs\ANN_fgsm\processed_images\test\img_37393.png
Missing image: ..\..\datasets\adv_outputs\ANN_fgsm\processed_images\test\img_37394.png
Missing image: ..\..\datasets\adv_outputs\ANN_fgsm\processed_images\test\img_37395.png
Missing image: ..\..\datasets\adv_outputs\ANN_fgsm\processed_images\test\img_37396.png
Missing image: ..\..\datasets\adv_outputs\ANN_fgsm\processed_images\test\img_37397.png


100%|██████████| 43299/43299 [03:06<00:00, 232.51it/s] 


Shapes:
X_train_img (92121, 50, 50, 3) X_train_meta (92121, 1) y_train (92121,)
X_val_img (18852, 50, 50, 3) X_val_meta (18852, 1) y_val (18852,)
X_test_img (37631, 50, 50, 3) X_test_meta (37631, 1) y_test (37631,)


4. ANN Baseline (Flattened + isNight)

In [5]:
# Build a simple feed-forward ANN for binary detection (clean=0 / adversarial=1).
# Inputs = flattened image pixels concatenated with the 1D aux feature `isNight`.
# Loss = binary_crossentropy, final activation = sigmoid (binary output).
# Metrics include accuracy, precision, and recall.

# Flatten images: (N, H, W, C) -> (N, H*W*C)
X_train_flat = X_train_img.reshape((X_train_img.shape[0], -1))
X_val_flat   = X_val_img.reshape((X_val_img.shape[0], -1))
X_test_flat  = X_test_img.reshape((X_test_img.shape[0], -1))

# Concatenate the aux feature `isNight` as the last column
X_train_flat = np.hstack([X_train_flat, X_train_meta])
X_val_flat   = np.hstack([X_val_flat, X_val_meta])
X_test_flat  = np.hstack([X_test_flat, X_test_meta])

# Define a compact ANN baseline
ann_model = Sequential([
    Input(shape=(X_train_flat.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile with binary loss and standard classification metrics
ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])

# Model summary for verification
ann_model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               960256    
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dense_2 (Dense)             (None, 1)                 65        
                                                                 
Total params: 968577 (3.69 MB)
Trainable params: 968577 (3.69 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


5. Train ANN Baseline (Weighted, Early Stop, Checkpoint)

In [7]:
# Train the ANN detector with class weighting to mitigate imbalance.
# Use EarlyStopping to avoid overfitting and restore the best weights.
# Save the best model (lowest val_loss) via ModelCheckpoint.
# class_weight.compute_class_weight balances clean vs adversarial classes.
# shuffle=True to decorrelate batches across epochs.

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True, 
    verbose=1
)

checkpoint = ModelCheckpoint(
    '../../models/ann_detector.keras', 
    monitor='val_loss', 
    save_best_only=True, 
    verbose=1
)

# Compute per-class weights (handles class imbalance)
cw = class_weight.compute_class_weight(
    'balanced', 
    classes=np.unique(y_train), 
    y=y_train
)

cw_dict = {int(i): float(w) for i,w in enumerate(cw)}
print('Class weights:', cw_dict)

# Train ANN
history_ann = ann_model.fit(
    X_train_flat, y_train, 
    validation_data=(X_val_flat, y_val), 
    epochs=NUM_EPOCHS, 
    batch_size=BATCH_SIZE, 
    callbacks=[early_stop, checkpoint], 
    class_weight=cw_dict, 
    shuffle=True
)

Class weights: {0: 1.5, 1: 0.75}
Epoch 1/30


1433/1440 [============================>.] - ETA: 0s - loss: 0.1127 - accuracy: 0.9557 - precision: 0.9828 - recall: 0.9502
Epoch 1: val_loss improved from inf to 0.05891, saving model to ../../models\ann_detector.keras
1440/1440 [==============================] - 12s 7ms/step - loss: 0.1124 - accuracy: 0.9559 - precision: 0.9829 - recall: 0.9504 - val_loss: 0.0589 - val_accuracy: 0.9785 - val_precision: 0.9750 - val_recall: 0.9921
Epoch 2/30
1440/1440 [==============================] - ETA: 0s - loss: 0.0353 - accuracy: 0.9891 - precision: 0.9952 - recall: 0.9885
Epoch 2: val_loss did not improve from 0.05891
1440/1440 [==============================] - 10s 7ms/step - loss: 0.0353 - accuracy: 0.9891 - precision: 0.9952 - recall: 0.9885 - val_loss: 0.0910 - val_accuracy: 0.9743 - val_precision: 0.9709 - val_recall: 0.9898
Epoch 3/30
1438/1440 [============================>.] - ETA: 0s - loss: 0.0281 - accuracy: 0.9912 - precision: 0.9961 - 

6. CNN Binary Model with Meta Input (isNight)

In [8]:
# Define a lightweight CNN for binary detection (clean=0 / adversarial=1).
# Combine image features with a single meta feature (isNight) via a second input.
# Conv blocks → GlobalAveragePooling → Dense.
# Meta branch: small MLP on isNight → concatenation → Dropout → sigmoid.
# Compiled with binary cross-entropy and accuracy/precision/recall metrics.
# NUM_CHANNELS should match your image tensors (3 for RGB/BGR).
# Keep IMG_SIZE consistent with preprocessing.

def build_cnn_binary(input_shape=(IMG_SIZE, IMG_SIZE, NUM_CHANNELS)):
    img_in = Input(shape=input_shape, name='image')
    x = Conv2D(32, 3, activation='relu', padding='same')(img_in)
    x = MaxPool2D()(x)
    x = Conv2D(64, 3, activation='relu', padding='same')(x)
    x = MaxPool2D()(x)
    x = Conv2D(128, 3, activation='relu', padding='same')(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(64, activation='relu')(x)

    
    # Meta branch for isNight (shape=(None, 1))
    meta_in = Input(shape=(1,), name='isNight')
    m = Dense(8, activation='relu')(meta_in)

    # Fuse image + meta features
    combined = concatenate([x, m])
    combined = Dropout(0.3)(combined)

    # Binary output (clean/adversarial)
    out = Dense(1, activation='sigmoid')(combined)

    model = Model(inputs=[img_in, meta_in], outputs=out, name='cnn_binary')
    model.compile(
        optimizer='adam', 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
    )
    return model

cnn_model = build_cnn_binary(input_shape=(IMG_SIZE, IMG_SIZE, NUM_CHANNELS))
cnn_model.summary()



Model: "cnn_binary"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 image (InputLayer)          [(None, 50, 50, 3)]          0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 50, 50, 32)           896       ['image[0][0]']               
                                                                                                  
 max_pooling2d (MaxPooling2  (None, 25, 25, 32)           0         ['conv2d[0][0]']              
 D)                                                                                               
                                                                                                  
 conv2d_1 (Conv2D)           (None, 25, 25, 64)           18496     ['max_pooling2d[0][0

7. Train CNN Detector (Image + isNight)

In [9]:
# Train the CNN-based binary detector using image tensors and the meta feature (isNight).
# Save the checkpoint based on validation loss; use early stopping from the ANN cell.
# X_train_img / X_val_img: image tensors (N, IMG_SIZE, IMG_SIZE, NUM_CHANNELS)
# X_train_meta / X_val_meta: meta tensors (N, 1) with isNight∈{0,1}
# y_train / y_val: binary labels (0=clean, 1=adversarial)
# cw_dict applies class weighting to mitigate class imbalance.
# early_stop is reused to restore the weights when validation loss stops improving

checkpoint_cnn = ModelCheckpoint(
    '../../models/cnn_detector.keras', 
    monitor='val_loss', 
    save_best_only=True, 
    verbose=1
)

history_cnn = cnn_model.fit(
    [X_train_img, X_train_meta], y_train, 
    validation_data=([X_val_img, X_val_meta], y_val), 
    epochs=NUM_EPOCHS, 
    batch_size=BATCH_SIZE, 
    callbacks=[early_stop, checkpoint_cnn], 
    class_weight=cw_dict, 
    shuffle=True
)

Epoch 1/30
1439/1440 [============================>.] - ETA: 0s - loss: 0.0204 - accuracy: 0.9930 - precision: 0.9936 - recall: 0.9960
Epoch 1: val_loss improved from inf to 0.00085, saving model to ../../models\cnn_detector.keras
1440/1440 [==============================] - 53s 36ms/step - loss: 0.0204 - accuracy: 0.9930 - precision: 0.9936 - recall: 0.9960 - val_loss: 8.5415e-04 - val_accuracy: 0.9997 - val_precision: 0.9998 - val_recall: 0.9997
Epoch 2/30
1439/1440 [============================>.] - ETA: 0s - loss: 0.0071 - accuracy: 0.9981 - precision: 0.9996 - recall: 0.9976
Epoch 2: val_loss did not improve from 0.00085
1440/1440 [==============================] - 53s 37ms/step - loss: 0.0071 - accuracy: 0.9981 - precision: 0.9996 - recall: 0.9976 - val_loss: 0.0013 - val_accuracy: 0.9998 - val_precision: 0.9998 - val_recall: 0.9999
Epoch 3/30
1440/1440 [==============================] - ETA: 0s - loss: 0.0037 - accuracy: 0.9992 - precision: 0.9999 - recall: 0.9989
Epoch 3: val_l

8. RNN (BiLSTM) Binary Detector (Image → Sequence + isNight)

In [10]:
# Convert 2D images into per-row sequences and train a BiLSTM-based binary detector.
# Input 1: sequence of length IMG_SIZE with (IMG_SIZE * NUM_CHANNELS) features per step.
# Input 2: meta feature isNight (0/1), concatenated before the final classifier.
# image_to_sequence reshapes (N, H, W, C) → (N, H, W*C) to emulate a scanline sequence.
# Model outputs a single sigmoid unit: 0 = clean, 1 = adversarial.
# Uses accuracy, precision, recall as training metrics.

def image_to_sequence(imgs):
    """Reshape images (N, H, W, C) into sequences (N, H, W*C)."""
    N = imgs.shape[0]
    return imgs.reshape((N, IMG_SIZE, IMG_SIZE * NUM_CHANNELS))

def build_rnn_binary(time_steps=IMG_SIZE, features=IMG_SIZE*NUM_CHANNELS):
    """BiLSTM-based binary classifier with an auxiliary meta input (isNight)."""
    seq_in = Input(shape=(time_steps, features), name='sequence')
    x = Bidirectional(LSTM(64))(seq_in)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)

    meta_in = Input(shape=(1,), name='isNight')
    m = Dense(8, activation='relu')(meta_in)

    combined = concatenate([x, m])
    out = Dense(1, activation='sigmoid')(combined)

    model = Model(inputs=[seq_in, meta_in], outputs=out, name='rnn_binary')
    model.compile(
        optimizer='adam', 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])
    return model

# Build sequence tensors for each split
X_train_seq = image_to_sequence(X_train_img)
X_val_seq   = image_to_sequence(X_val_img)
X_test_seq  = image_to_sequence(X_test_img)

# Instantiate and inspect the model
rnn_model = build_rnn_binary()
rnn_model.summary()

Model: "rnn_binary"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 sequence (InputLayer)       [(None, 50, 150)]            0         []                            
                                                                                                  
 bidirectional (Bidirection  (None, 128)                  110080    ['sequence[0][0]']            
 al)                                                                                              
                                                                                                  
 dropout_1 (Dropout)         (None, 128)                  0         ['bidirectional[0][0]']       
                                                                                                  
 isNight (InputLayer)        [(None, 1)]                  0         []                   

9. Train RNN Binary Detector

In [11]:
# Train the BiLSTM-based binary detector (clean=0 / adversarial=1).
# Use early stopping and model checkpointing on validation loss.
# Class weights address any clean/adv imbalance.

checkpoint_rnn = ModelCheckpoint(
    '../../models/rnn_detector.keras', 
    monitor='val_loss', 
    save_best_only=True, verbose=1
)

history_rnn = rnn_model.fit(
    [X_train_seq, X_train_meta], y_train, 
    validation_data=([X_val_seq, X_val_meta], y_val), 
    epochs=NUM_EPOCHS, 
    batch_size=BATCH_SIZE, 
    callbacks=[early_stop, checkpoint_rnn], 
    class_weight=cw_dict, 
    shuffle=True
)


Epoch 1/30
1438/1440 [============================>.] - ETA: 0s - loss: 0.1392 - accuracy: 0.9385 - precision: 0.9716 - recall: 0.9352
Epoch 1: val_loss improved from inf to 0.05476, saving model to ../../models\rnn_detector.keras
1440/1440 [==============================] - 40s 25ms/step - loss: 0.1391 - accuracy: 0.9386 - precision: 0.9716 - recall: 0.9352 - val_loss: 0.0548 - val_accuracy: 0.9833 - val_precision: 0.9830 - val_recall: 0.9914
Epoch 2/30
1438/1440 [============================>.] - ETA: 0s - loss: 0.0352 - accuracy: 0.9875 - precision: 0.9946 - recall: 0.9867
Epoch 2: val_loss improved from 0.05476 to 0.03701, saving model to ../../models\rnn_detector.keras
1440/1440 [==============================] - 34s 24ms/step - loss: 0.0353 - accuracy: 0.9875 - precision: 0.9946 - recall: 0.9866 - val_loss: 0.0370 - val_accuracy: 0.9861 - val_precision: 0.9847 - val_recall: 0.9939
Epoch 3/30
1440/1440 [==============================] - ETA: 0s - loss: 0.0164 - accuracy: 0.9944 - 

10. Evaluation (Reports + Confusion Matrices)

In [12]:
# Load detector checkpoints for ANN/CNN/RNN (if present).
# Run inference on the held-out test set.
# Print sklearn classification reports and confusion matrices.
# Binary detection task (clean=0, adv=1) → sigmoid outputs thresholded at 0.5.
# Checkpoint filenames:
# ANN: 'ann_detector.keras'
# CNN: 'cnn_detector.keras'
# RNN: saved earlier as 'rnn_detector.keras' (change below if you used a different name).

BATCH_EVAL = 512  # adjust if you hit memory issues

def _to_labels(y):
    """Accepts one-hot/proba or raw ints and returns hard class indices."""
    y = np.asarray(y)
    return np.argmax(y, axis=1) if y.ndim == 2 else y.ravel()

# Ground-truth (expects y_test as 0/1 integers or one-hot) 
y_true = _to_labels(y_test)

# Load checkpoints if available
if os.path.exists('../../models/ann_detector.keras'):
    ann_model.load_weights('../../models/ann_detector.keras')
if os.path.exists('../../models/cnn_detector.keras'):
    cnn_model.load_weights('../../models/cnn_detector.keras')

# RNN: adjust filename if you saved 'rnn_detector.keras' instead
rnn_ckpt = '../../models/rnn_detector.keras'
if os.path.exists(rnn_ckpt):
    rnn_model.load_weights(rnn_ckpt)

# Evaluate ANN 
try:
    y_pred = ann_model.predict(X_test_flat, batch_size=BATCH_EVAL)
    y_pred_ann = (y_pred.ravel() > 0.5).astype(int) if y_pred.ndim == 2 and y_pred.shape[1] == 1 else np.argmax(y_pred, axis=1)
    print("=== ANN ===")
    print(classification_report(y_true, y_pred_ann, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred_ann))
except Exception as e:
    print("ANN eval failed:", e)

# Evaluate CNN 
try:
    y_pred = cnn_model.predict([X_test_img, X_test_meta], batch_size=BATCH_EVAL)
    y_pred_cnn = (y_pred.ravel() > 0.5).astype(int) if y_pred.ndim == 2 and y_pred.shape[1] == 1 else np.argmax(y_pred, axis=1)
    print("\n=== CNN ===")
    print(classification_report(y_true, y_pred_cnn, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred_cnn))
except Exception as e:
    print("CNN eval failed:", e)

# Evaluate RNN 
try:
    y_pred = rnn_model.predict([X_test_seq, X_test_meta], batch_size=BATCH_EVAL)
    y_pred_rnn = (y_pred.ravel() > 0.5).astype(int) if y_pred.ndim == 2 and y_pred.shape[1] == 1 else np.argmax(y_pred, axis=1)
    print("\n=== RNN ===")
    print(classification_report(y_true, y_pred_rnn, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred_rnn))
except Exception as e:
    print("RNN eval failed:", e)


74/74 [==============================] - 1s 5ms/step
=== ANN ===
              precision    recall  f1-score   support

           0     0.9817    0.6931    0.8125     14433
           1     0.8386    0.9919    0.9088     23198

    accuracy                         0.8773     37631
   macro avg     0.9101    0.8425    0.8607     37631
weighted avg     0.8935    0.8773    0.8719     37631

Confusion matrix:
 [[10004  4429]
 [  187 23011]]
74/74 [==============================] - 7s 93ms/step

=== CNN ===
              precision    recall  f1-score   support

           0     0.9985    0.9997    0.9991     14433
           1     0.9998    0.9991    0.9995     23198

    accuracy                         0.9993     37631
   macro avg     0.9992    0.9994    0.9993     37631
weighted avg     0.9993    0.9993    0.9993     37631

Confusion matrix:
 [[14429     4]
 [   21 23177]]
74/74 [==============================] - 4s 49ms/step

=== RNN ===
              precision    recall  f1-score   s